# Binary grain vs background (SwinV2 + UPerNet, 5-fold CV)

This notebook is separate from `colab_run_pipeline_221.ipynb`. It trains **binary** segmentation:

- **Class 0 (non-grain / background):** original multiclass label `0`.
- **Class 1 (grain):** any original label in `1`–`15` (bivalves, micrite, cement, …, brachiopod).
- **Ignore:** original value `255` stays ignored in the loss.

Multiclass masks are converted **on the fly** in the dataset (no duplicate mask files written).

Training uses the same **SwinV2-Tiny + UPerNet** setup as the multiclass pipeline (`num_labels=2`), **5-fold** cross-validation over image/mask pairs, and logs **pixel accuracy**, **mean IoU**, and **per-class IoU** each validation epoch. Summary metrics are saved to `cv_summary.json` under `--output_dir`.

In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Repo on Drive (update if your clone lives elsewhere)
REPO_ROOT = Path('/content/drive/My Drive/Payne_lab_swin_transformer')

IMG_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/img'
MASK_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/masks_machine'

# Outputs for binary CV runs (separate from multiclass OUT_ROOT in the other notebook)
OUT_BINARY = '/content/drive/My Drive/Petrographic images_ML work/model_outputs_binary_cv'

print('Repo exists:', REPO_ROOT.exists())
print('IMG dir exists:', Path(IMG_DIR).exists())
print('MASK dir exists:', Path(MASK_DIR).exists())

## Smoke test

Builds dataloaders and checks paired samples (no training).

In [ ]:
import subprocess
import sys
from pathlib import Path

script = REPO_ROOT / "code" / "model_training_pipeline" / "swin_binary_segmentation_221.py"
if not script.is_file():
    raise FileNotFoundError(f"Script not found (check REPO_ROOT): {script}")

argv = [
    sys.executable,
    str(script),
    "--img_dir",
    IMG_DIR,
    "--mask_dir",
    MASK_DIR,
    "--output_dir",
    f"{OUT_BINARY}/smoke",
    "--no_train",
]
print("Running:", " ".join(argv))
proc = subprocess.run(argv, cwd=str(REPO_ROOT))
print("\nExit code:", proc.returncode, "(0 = success; 2 often means Python could not run the script or argparse error)")

## Full 5-fold cross-validation

Runs **five** trainings (one per fold). Each fold writes `fold_k/best_upernet_swinv2_binary.pth`, `fold_k/val_metrics.csv`, and a combined `cv_summary.json` under `OUT_BINARY`.

Run the **config** cell (`REPO_ROOT`, `IMG_DIR`, …) before this cell. If you only see a number like `512` under the cell, that was the old `os.system` status word (e.g. exit code 2 = failed fast); the cell below now streams logs and prints a clear exit code.

Optional SSL init: if `SSL_BACKBONE` exists on Drive, `--backbone_checkpoint` is added automatically.

In [ ]:
import subprocess
import sys
from pathlib import Path

script = REPO_ROOT / "code" / "model_training_pipeline" / "swin_binary_segmentation_221.py"
if not script.is_file():
    raise FileNotFoundError(f"Script not found (check REPO_ROOT): {script}")

SSL_BACKBONE = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_221/ssl_full/ssl_swinv2_best.pth"
use_ssl = Path(SSL_BACKBONE).is_file()
print("SSL backbone:", "ON" if use_ssl else "OFF (ImageNet-only backbone)")

argv = [
    sys.executable,
    str(script),
    "--img_dir",
    IMG_DIR,
    "--mask_dir",
    MASK_DIR,
    "--output_dir",
    f"{OUT_BINARY}/cv5",
    "--n_folds",
    "5",
    "--epochs",
    "20",
    "--batch_size",
    "2",
    "--crop",
    "512",
    "--num_workers",
    "2",
]
if use_ssl:
    argv.extend(["--backbone_checkpoint", SSL_BACKBONE])

print("Running:", " ".join(argv))
proc = subprocess.run(argv, cwd=str(REPO_ROOT))
print("\nExit code:", proc.returncode, "(0 = success)")
if proc.returncode != 0:
    print("Training did not run to completion. Scroll up for Python stderr (traceback or argparse message).")